In [9]:
import torch
import open3d as o3d
import numpy as np
from pytorch3d.ops import sample_points_from_meshes
from pytorch3d.structures import Meshes
from pytorch3d.io import load_objs_as_meshes, save_obj
from pathlib import Path
import trimesh
#the result proof, resample the mesh can't gurantee the mesh to watertight
def make_mesh_2_watertight(trimesh_mesh):#this func can't gurantee make mesh to watertight
    trimesh_mesh.fill_holes()
    return trimesh_mesh

def convert_open3d_mesh_to_trimesh(open3d_mesh):
    # 从Open3D的mesh中获取顶点和面
    vertices = np.asarray(open3d_mesh.vertices)
    faces = np.asarray(open3d_mesh.triangles)
    
    # 创建一个trimesh的mesh对象
    trimesh_mesh = trimesh.Trimesh(vertices=vertices, faces=faces)
    
    return trimesh_mesh


def simplify_with_resample():
    folder = Path("/media/tony/T7/camera_data/test_object_position_optimize/")
    obj_name = Path("banana.obj")
    path = folder / obj_name
    # 如果您的 GPU 可用，指定 device 为 'cuda'
    device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

    # 加载网格
    mesh = load_objs_as_meshes([path], device=device)

    # 从网格中采样点云
    num_samples = 3000  # 你想从网格中采样多少点
    pointcloud = sample_points_from_meshes(mesh, num_samples)

    # 将点云转换回numpy格式以便Open3D处理
    points = pointcloud.cpu().numpy()[0]
    point_cloud_o3d = o3d.geometry.PointCloud()
    point_cloud_o3d.points = o3d.utility.Vector3dVector(points)
    point_cloud_o3d.estimate_normals()

    # 使用Open3D重建网格
    distances = point_cloud_o3d.compute_nearest_neighbor_distance()
    avg_dist = np.mean(distances)
    radius = 3 * avg_dist  # Open3D中的半径选择是一个经验值

    # 通过ball pivoting算法重建网格
    mesh_reconstructed = o3d.geometry.TriangleMesh.create_from_point_cloud_ball_pivoting(
        point_cloud_o3d,
        o3d.utility.DoubleVector([radius, radius * 2])
    )
    mesh = convert_open3d_mesh_to_trimesh(mesh_reconstructed)
    mesh = make_mesh_2_watertight(mesh)
    # 保存重建的网格
    o3d.io.write_triangle_mesh(str(folder / f"simplify{obj_name}_two.obj"), mesh_reconstructed)
simplify_with_resample()

[Open3D WARNING] Write OBJ can not include triangle normals.
